In [1]:
"""
DREAMER (Valence) -- Multi-Scale Spatial-Temporal Masked Autoencoder (Fast Single-Seed FP32)
==========================================================================================
1. Target: Valence Binary Classification (ScoreValence > 3.0 = High, <= 3.0 = Low).
2. Platform: Strictly Kaggle (/kaggle/input and /kaggle/working).
3. Data Preprocessing & Two-Tier Normalization:
   - Per-video baseline subtraction: (feat - baseline_mean) using DREAMER.mat.
   - Global feature Z-score standardization: (X - mu) / sd across the 10 frequency bands.
4. Architecture:
   - 14-channel 10-20 ScalpPositionalEncoding mapped to the Emotiv EPOC montage.
   - Globally pretrained STMAE with topological masking.
   - MSC-TimesNet with FFT Top-3 period extraction & multi-scale convolutions.
   - Regularization: Channel Dropout (0.1), Mixup (alpha=0.2), AdaBN test recalibration.
5. Execution & Metrics:
   - Single seed: SEED = 42 (23 LOSO folds).
   - In-memory GPU VRAM pre-loading (X_pt) + cuDNN auto-tuning.
   - Full metric suite: Win/Trial Accuracy, Balanced Accuracy, Macro-F1, Weighted-F1.
"""

import os
import time
import math
import random
import copy
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import welch, butter, filtfilt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# =====================================================================
# CONFIGURATION & PATHS (KAGGLE STRICT)
# =====================================================================
OUTPUT_DIR = "/kaggle/working/dreamer_valence_stmae_timesnet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAT_SEARCH_PATHS = [
    "/kaggle/input/datasets/gautam2411/dreamer2",
    "/kaggle/input/dreamer/DREAMER.mat",
    "/kaggle/input/dreamer2/DREAMER.mat",
    "./dreamer_features/X_raw.npy"
]

N_SUBJECTS = 23
N_VIDEOS = 18
NUM_CHANNELS = 14
RAW_DIM = 10            # 5 PSD + 5 DE bands
NUM_CLASSES = 2         # Binary: Low Valence (0) vs High Valence (1)
THRESHOLD = 3.0         # Paper rating threshold (> 3.0 = High)
CLASS_NAMES = ["Low Valence", "High Valence"]

# ---- Stage A : STMAE ----
EMBEDDING_SIZE = 32
AUTOENCODER_HIDDEN_SIZE = 64
AUTOENCODER_HEADS = 4
AUTOENCODER_LAYERS = 3
AUTOENCODER_EPOCHS = 30
AUTOENCODER_LR = 1e-3
AUTOENCODER_BATCH_SIZE = 256
RANDOM_MASK_FRACTION = 0.40
REGION_MASK_CHANCE = 0.60

# ---- Stage B : Sequences ----
WINDOW_LENGTH = 10      # 10 consecutive 0.5s windows (5-second window context)
STRIDE = 1              # Dense window overlap

# ---- Stage C : MSC-TimesNet ----
CLASSIFIER_HIDDEN_SIZE = 128
CLASSIFIER_BLOCKS = 2
TOP_FREQUENCIES = 3
CLASSIFIER_FEEDFORWARD_SIZE = 256
CLASSIFIER_HEADS = 4
CLASSIFIER_DROPOUT = 0.3

# ---- Training Dynamics ----
MAX_EPOCHS = 50
MIN_EPOCHS = 15
PATIENCE_EPOCHS = 10
LEARNING_RATE = 1e-3
ENCODER_LR_SCALE = 0.1
BATCH_SIZE = 128
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 5
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
VALIDATION_FRACTION = 0.2
MIXUP_STRENGTH = 0.2
CHANNEL_DROPOUT_RATE = 0.1

RECALIBRATE_BATCHNORM = True
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 14 Channels corresponding to Emotiv EPOC standard montage in DREAMER
CHANNEL_NAMES = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"
]
assert len(CHANNEL_NAMES) == NUM_CHANNELS

REPORT = []


def seed_everything(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =====================================================================
# SCALP GEOMETRY (10-20 layout for DREAMER's 14 electrodes)
# =====================================================================
def build_scalp_coords():
    coords = {
        "AF3": (-0.40, 0.75), "AF4": (0.40, 0.75),
        "F7":  (-0.75, 0.50), "F3":  (-0.35, 0.50), "F4": (0.35, 0.50), "F8": (0.75, 0.50),
        "FC5": (-0.65, 0.25), "FC6": (0.65, 0.25),
        "T7":  (-0.85, 0.00), "T8":  (0.85, 0.00),
        "P7":  (-0.70, -0.45), "P8": (0.70, -0.45),
        "O1":  (-0.30, -0.80), "O2": (0.30, -0.80)
    }
    return np.array([coords[c] for c in CHANNEL_NAMES], dtype=np.float32)


def build_regions():
    coords = build_scalp_coords()
    regions = defaultdict(list)
    for i, name in enumerate(CHANNEL_NAMES):
        x, y = coords[i]
        if y >= 0.40:
            key = "frontal"
        elif abs(x) > 0.60 and -0.20 <= y < 0.40:
            key = "temporal_left" if x < 0 else "temporal_right"
        elif y <= -0.60:
            key = "occipital"
        else:
            key = "parietal"
        regions[key].append(i)
    return {k: np.array(v, dtype=np.int64) for k, v in regions.items()}


SCALP_COORDS = build_scalp_coords()
REGIONS = build_regions()


# =====================================================================
# DATA EXTRACTION & TWO-TIER NORMALIZATION (VALENCE)
# =====================================================================
EEG_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}
FS = 128
WIN_SAMP = 64  # 0.5 sec at 128 Hz

# Integration helper compatible across NumPy versions (1.x and 2.x)
trapz_fn = getattr(np, "trapezoid", getattr(np, "trapz", None))


def bandpass_filter(signal, lowcut, highcut, fs=FS, order=4):
    nyquist = 0.5 * fs
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype="band")
    return filtfilt(b, a, signal)


def prefilter_trial(eeg):
    filtered = {}
    for band, (lo, hi) in EEG_BANDS.items():
        band_sig = np.zeros_like(eeg, dtype=np.float32)
        for ch in range(NUM_CHANNELS):
            band_sig[:, ch] = bandpass_filter(eeg[:, ch].astype(np.float64), lo, hi)
        filtered[band] = band_sig
    return filtered


def window_features(raw_window, filtered_windows):
    feat = np.zeros((NUM_CHANNELS, RAW_DIM), dtype=np.float32)
    freqs, psd = welch(raw_window.T, fs=FS, axis=-1, nperseg=WIN_SAMP, noverlap=0)
    for ch in range(NUM_CHANNELS):
        psd_feat, de_feat = [], []
        for band, (lo, hi) in EEG_BANDS.items():
            mask = (freqs >= lo) & (freqs < hi)
            power = trapz_fn(psd[ch, mask], freqs[mask]) if np.any(mask) else 0.0
            psd_feat.append(power)
            filt = filtered_windows[band][:, ch]
            var = max(float(np.var(filt)), 1e-10)
            de_feat.append(0.5 * np.log2(2 * np.pi * np.e * var))
        feat[ch, :5] = psd_feat
        feat[ch, 5:] = de_feat
    return feat


def load_or_extract_dreamer_valence():
    x_cache = os.path.join(OUTPUT_DIR, "X_raw.npy")
    y_cache = os.path.join(OUTPUT_DIR, "Y_valence.npy")
    sub_cache = os.path.join(OUTPUT_DIR, "subject_ids.npy")
    tri_cache = os.path.join(OUTPUT_DIR, "trial_ids.npy")

    if (os.path.exists(x_cache) and os.path.exists(y_cache) and 
        os.path.exists(sub_cache) and os.path.exists(tri_cache)):
        print("  Loading pre-extracted features and valence labels from cache...")
        X_raw = np.load(x_cache)
        Y = np.load(y_cache)
        subject_ids = np.load(sub_cache)
        trial_ids = np.load(tri_cache)
        return X_raw, Y, subject_ids, trial_ids

    # Locate DREAMER.mat
    mat_file = None
    for p in MAT_SEARCH_PATHS:
        if os.path.exists(p) and p.endswith(".mat"):
            mat_file = p
            break
        elif os.path.isdir(p):
            cand = os.path.join(p, "DREAMER.mat")
            if os.path.exists(cand):
                mat_file = cand
                break

    if mat_file is None:
        raise FileNotFoundError("Could not find DREAMER.mat in /kaggle/input/. Check your dataset mount path.")

    print(f"  Parsing raw EEG from {mat_file} (Welch PSD + DE + Baseline Subtraction for Valence)...")
    mat = sio.loadmat(mat_file, simplify_cells=False)
    dreamer = mat["DREAMER"][0, 0]
    data_all = dreamer["Data"][0]

    all_feats, all_labels, all_subj, all_trial = [], [], [], []

    def get_video_eeg(field_cell, vid_idx):
        arr = field_cell[0, 0]
        if isinstance(arr, np.ndarray) and arr.dtype == object:
            arr = arr.flat[vid_idx]
        if arr.shape[0] == NUM_CHANNELS:
            arr = arr.T
        return np.asarray(arr, dtype=np.float32)

    for subj_idx in range(N_SUBJECTS):
        subj_data = data_all[subj_idx]
        eeg_data = subj_data["EEG"][0, 0]
        # Target: ScoreValence
        scores_val = subj_data["ScoreValence"][0, 0].flatten()
        stimuli_cell = eeg_data["stimuli"]
        baseline_cell = eeg_data["baseline"]

        for vid_idx in range(N_VIDEOS):
            bl_eeg = get_video_eeg(baseline_cell, vid_idx)
            bl_filtered = prefilter_trial(bl_eeg)
            baseline_features = []
            for start in range(0, bl_eeg.shape[0] - WIN_SAMP + 1, WIN_SAMP):
                r_win = bl_eeg[start:start + WIN_SAMP, :]
                f_win = {b: bl_filtered[b][start:start + WIN_SAMP] for b in EEG_BANDS}
                baseline_features.append(window_features(r_win, f_win))
            baseline_mean = np.stack(baseline_features).mean(axis=0)

            emo_eeg = get_video_eeg(stimuli_cell, vid_idx)
            emo_filtered = prefilter_trial(emo_eeg)
            label = float(scores_val[vid_idx])

            for start in range(0, emo_eeg.shape[0] - WIN_SAMP + 1, WIN_SAMP):
                r_win = emo_eeg[start:start + WIN_SAMP, :]
                f_win = {b: emo_filtered[b][start:start + WIN_SAMP] for b in EEG_BANDS}
                feat = window_features(r_win, f_win)
                # Baseline subtraction normalization
                corrected = feat - baseline_mean
                all_feats.append(corrected)
                all_labels.append(label)
                all_subj.append(subj_idx)
                all_trial.append(vid_idx)

    X_raw = np.stack(all_feats).astype(np.float32)
    Y = np.array(all_labels, dtype=np.float32)
    subject_ids = np.array(all_subj, dtype=np.int32)
    trial_ids = np.array(all_trial, dtype=np.int32)

    np.save(x_cache, X_raw)
    np.save(y_cache, Y)
    np.save(sub_cache, subject_ids)
    np.save(tri_cache, trial_ids)

    return X_raw, Y, subject_ids, trial_ids


def normalize_dreamer_global(X_raw):
    """Z-score on flattened features matching reference preprocessing."""
    orig_shape = X_raw.shape
    X_flat = X_raw.reshape(-1, RAW_DIM).astype(np.float32)
    mu = X_flat.mean(axis=0, keepdims=True)
    sd = X_flat.std(axis=0, keepdims=True)
    sd[sd < 1e-8] = 1.0
    X_norm = (X_flat - mu) / sd
    return X_norm.reshape(orig_shape)


# =====================================================================
# STAGE A : STMAE (14-ELECTRODE GEOMETRY)
# =====================================================================
class ScalpPositionalEncoding(nn.Module):
    def __init__(self, coords, d_model):
        super().__init__()
        self.register_buffer("coords", torch.tensor(coords, dtype=torch.float32))
        self.mlp = nn.Sequential(nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model))

    def forward(self, x):
        return x + self.mlp(self.coords).unsqueeze(0)


class STMAE(nn.Module):
    def __init__(self, in_feat, d_model=AUTOENCODER_HIDDEN_SIZE, latent_dim=EMBEDDING_SIZE,
                 heads=AUTOENCODER_HEADS, layers=AUTOENCODER_LAYERS):
        super().__init__()
        self.proj = nn.Linear(in_feat, d_model)
        self.pos = ScalpPositionalEncoding(SCALP_COORDS, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.mask_token, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.to_latent = nn.Linear(d_model, latent_dim)
        self.latent_norm = nn.LayerNorm(latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, d_model), nn.GELU(), nn.Linear(d_model, in_feat)
        )

    def encode(self, x, mask=None):
        h = self.proj(x)
        if mask is not None:
            h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
        h = self.pos(h)
        h = self.encoder(h)
        return self.latent_norm(self.to_latent(h))

    def forward(self, x, mask):
        z = self.encode(x, mask)
        return self.decoder(z), z


def sample_mask(batch_size, device):
    mask = torch.zeros(batch_size, NUM_CHANNELS, dtype=torch.bool, device=device)
    region_keys = list(REGIONS.keys())
    for b in range(batch_size):
        if random.random() < REGION_MASK_CHANCE:
            k = random.choice([1, 2])
            for key in random.sample(region_keys, k):
                mask[b, torch.tensor(REGIONS[key], device=device)] = True
        else:
            n = max(1, int(RANDOM_MASK_FRACTION * NUM_CHANNELS))
            idx = torch.randperm(NUM_CHANNELS, device=device)[:n]
            mask[b, idx] = True
    return mask


def pretrain_stmae(X_pt, epochs=AUTOENCODER_EPOCHS, lr=AUTOENCODER_LR, batch=AUTOENCODER_BATCH_SIZE):
    in_feat = X_pt.shape[2]
    model = STMAE(in_feat).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    n = X_pt.shape[0]
    model.train()
    for ep in range(1, epochs + 1):
        perm = torch.randperm(n)
        tot, nb = 0.0, 0
        for i in range(0, n, batch):
            xb = X_pt[perm[i:i + batch]]
            mask = sample_mask(xb.shape[0], DEVICE)
            recon, _ = model(xb, mask)
            m = mask.unsqueeze(-1).expand_as(xb)
            loss = F.mse_loss(recon[m], xb[m])
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            opt.step()
            tot += loss.item()
            nb += 1
        sched.step()
        if ep == 1 or ep % 5 == 0:
            print("  AE ep%03d masked_recon_mse=%.5f" % (ep, tot / max(nb, 1)))
    torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "stmae_global.pt"))
    return model


# =====================================================================
# STAGE B : SEQUENCE BUILDER
# =====================================================================
def build_sequence_index(y, subs, tri, seq_len=WINDOW_LENGTH, stride=STRIDE):
    keys = subs * 1000 + tri
    seqs, labs, s_sub, s_tri = [], [], [], []
    order = np.argsort(keys, kind="stable")
    for k in np.unique(keys):
        rows = order[keys[order] == k]
        if rows.shape[0] < seq_len:
            continue
        for start in range(0, rows.shape[0] - seq_len + 1, stride):
            win = rows[start:start + seq_len]
            seqs.append(win)
            labs.append(y[win[0]])
            s_sub.append(subs[win[0]])
            s_tri.append(tri[win[0]])
    return (np.asarray(seqs, dtype=np.int64), np.asarray(labs, dtype=np.int64),
            np.asarray(s_sub, dtype=np.int64), np.asarray(s_tri, dtype=np.int64))


# =====================================================================
# STAGE C : MSC-TimesNet
# =====================================================================
def fft_topk_periods(x, k=TOP_FREQUENCIES):
    B, T, d = x.shape
    xf = torch.fft.rfft(x, dim=1)
    amp = xf.abs().mean(dim=2)
    amp[:, 0] = 0.0
    k = min(k, max(amp.shape[1] - 1, 1))
    _, idx = torch.topk(amp, k, dim=1)
    freqs = idx.float().mean(dim=0).round().long().clamp(min=1)
    periods = [max(int(T // f.item()), 1) for f in freqs]
    weights = torch.stack([amp[:, i] for i in freqs], dim=1)
    return periods, F.softmax(weights, dim=1)


class MultiScaleConvBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        h = max(d_model // 4, 8)
        self.b1 = nn.Sequential(nn.Conv2d(d_model, h, 1), nn.BatchNorm2d(h), nn.GELU())
        self.b3 = nn.Sequential(nn.Conv2d(d_model, h, 3, padding=1), nn.BatchNorm2d(h), nn.GELU())
        self.b5 = nn.Sequential(nn.Conv2d(d_model, h, 5, padding=2), nn.BatchNorm2d(h), nn.GELU())
        self.bp = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1), nn.Conv2d(d_model, h, 1),
            nn.BatchNorm2d(h), nn.GELU(),
        )
        self.fuse = nn.Sequential(nn.Conv2d(4 * h, d_model, 1), nn.BatchNorm2d(d_model))

    def forward(self, x):
        return self.fuse(torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1))


class TimesBlock(nn.Module):
    def __init__(self, d_model, topk=TOP_FREQUENCIES):
        super().__init__()
        self.topk = topk
        self.conv = MultiScaleConvBlock(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, d = x.shape
        periods, weights = fft_topk_periods(x, self.topk)
        outs = []
        for p in periods:
            pad = (math.ceil(T / p) * p) - T
            xp = F.pad(x, (0, 0, 0, pad)) if pad > 0 else x
            Tp = xp.shape[1]
            num_p = Tp // p
            z = xp.permute(0, 2, 1).reshape(B, d, num_p, p)
            z = self.conv(z)
            z = z.reshape(B, d, Tp).permute(0, 2, 1)[:, :T, :]
            outs.append(z)
        stacked = torch.stack(outs, dim=-1)
        w = weights.unsqueeze(1).unsqueeze(1)
        agg = (stacked * w).sum(dim=-1)
        return self.norm(agg + x)


class MSCTimesNet(nn.Module):
    def __init__(self, in_dim, d_model=CLASSIFIER_HIDDEN_SIZE, blocks=CLASSIFIER_BLOCKS,
                 num_classes=NUM_CLASSES, dropout=CLASSIFIER_DROPOUT):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(in_dim, d_model), nn.LayerNorm(d_model))
        self.blocks = nn.ModuleList([TimesBlock(d_model) for _ in range(blocks)])
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=CLASSIFIER_HEADS, dim_feedforward=CLASSIFIER_FEEDFORWARD_SIZE,
            dropout=dropout, batch_first=True, norm_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        h = self.inp(x)
        for blk in self.blocks:
            h = blk(h)
        h = self.transformer(h)
        return self.head(h.mean(dim=1))


class EndToEndModel(nn.Module):
    def __init__(self, stmae, in_feat, finetune_encoder=False):
        super().__init__()
        self.stmae = stmae
        self.finetune_encoder = finetune_encoder
        self.net = MSCTimesNet(NUM_CHANNELS * EMBEDDING_SIZE)
        self.set_encoder_trainable(finetune_encoder)

    def set_encoder_trainable(self, flag):
        for p in self.stmae.parameters():
            p.requires_grad = bool(flag)

    def encode_seq(self, x):
        B, T, C, Fq = x.shape
        flat = x.reshape(B * T, C, Fq)
        if self.finetune_encoder and self.training:
            z = self.stmae.encode(flat)
        else:
            with torch.no_grad():
                z = self.stmae.encode(flat)
        return z.reshape(B, T, C * EMBEDDING_SIZE)

    def forward(self, x):
        return self.net(self.encode_seq(x))


# =====================================================================
# FAST GPU UTILITIES
# =====================================================================
def balanced_weights(y):
    cnt = np.bincount(y, minlength=NUM_CLASSES).astype(np.float32)
    cnt[cnt == 0] = 1.0
    w = cnt.sum() / (NUM_CLASSES * cnt)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def lr_at(ep, base_lr, warmup_epochs=None):
    if warmup_epochs is None:
        warmup_epochs = globals().get("WARMUP_EPOCHS", 3)
    warmup_epochs = max(1, int(warmup_epochs))
    if ep <= warmup_epochs:
        return base_lr * ep / warmup_epochs
    prog = (ep - warmup_epochs) / max(1, MAX_EPOCHS - warmup_epochs)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))


def grouped_split(labels, groups, frac=VALIDATION_FRACTION, seed=RANDOM_SEED):
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    tr, va = next(gss.split(np.zeros(len(labels)), labels, groups))
    return tr, va


def make_batches(n, batch, shuffle=True):
    idx = np.random.permutation(n) if shuffle else np.arange(n)
    for i in range(0, n, batch):
        yield idx[i:i + batch]


def apply_channel_dropout(xb, p=CHANNEL_DROPOUT_RATE):
    if p <= 0:
        return xb
    B = xb.shape[0]
    keep = (torch.rand(B, 1, NUM_CHANNELS, 1, device=xb.device) > p).float()
    return xb * keep


def mixup(xb, yb, alpha=MIXUP_STRENGTH):
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(xb.shape[0], device=xb.device)
    return lam * xb + (1 - lam) * xb[perm], yb, yb[perm], lam


@torch.no_grad()
def adabn_recalibrate(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    had_bn = any(isinstance(m, nn.BatchNorm2d) for m in model.modules())
    if not had_bn:
        return model
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()
            m.momentum = None
            m.train()
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        model.net(model.encode_seq(xb))
    model.eval()
    return model


@torch.no_grad()
def predict_probs(model, X_pt, seq_idx_pt, rows_pt, batch=BATCH_SIZE):
    model.eval()
    out = []
    for b_rows in make_batches(len(rows_pt), batch, shuffle=False):
        xb = X_pt[seq_idx_pt[rows_pt[b_rows]]]
        out.append(F.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out, axis=0)


def fit(stmae, X_pt, seq_idx_pt, labels, groups, train_rows, seed):
    seed_everything(seed)
    in_feat = X_pt.shape[2]
    model = EndToEndModel(stmae, in_feat, finetune_encoder=True).to(DEVICE)
    tr_loc, va_loc = grouped_split(labels[train_rows], groups[train_rows], seed=seed)

    tr_rows_pt = torch.tensor(train_rows[tr_loc], dtype=torch.long, device=DEVICE)
    va_rows_pt = torch.tensor(train_rows[va_loc], dtype=torch.long, device=DEVICE)

    w = balanced_weights(labels[train_rows[tr_loc]])
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW([
        {"params": model.net.parameters(), "lr": LEARNING_RATE},
        {"params": model.stmae.parameters(), "lr": LEARNING_RATE * ENCODER_LR_SCALE},
    ], weight_decay=WEIGHT_DECAY)

    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, MAX_EPOCHS + 1):
        cur = lr_at(ep, LEARNING_RATE)
        for i, g in enumerate(opt.param_groups):
            g["lr"] = cur * (ENCODER_LR_SCALE if i == 1 else 1.0)

        model.train()
        for b_rows in make_batches(len(tr_rows_pt), BATCH_SIZE):
            b_idx = tr_rows_pt[b_rows]
            xb = X_pt[seq_idx_pt[b_idx]]
            yb = torch.tensor(labels[tr_rows_pt[b_rows].cpu().numpy()], dtype=torch.long, device=DEVICE)

            xb = apply_channel_dropout(xb)
            xb, ya, ybb, lam = mixup(xb, yb)
            logits = model(xb)
            loss = lam * crit(logits, ya) + (1 - lam) * crit(logits, ybb)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRADIENT_CLIP)
            opt.step()

        vp = predict_probs(model, X_pt, seq_idx_pt, va_rows_pt)
        vf1 = f1_score(labels[va_rows_pt.cpu().numpy()], vp.argmax(1), average="macro")
        if vf1 > best_f1:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if ep >= MIN_EPOCHS and bad >= PATIENCE_EPOCHS:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_f1


# =====================================================================
# EVALUATION & METRICS (VALENCE)
# =====================================================================
def trial_level_scores(probs, y, sub, tri):
    keys = sub * 1000 + tri
    yt, yp = [], []
    for k in np.unique(keys):
        m = keys == k
        yt.append(y[m][0])
        yp.append(probs[m].mean(axis=0).argmax())
    yt, yp = np.array(yt), np.array(yp)

    t_acc = accuracy_score(yt, yp)
    t_bacc = balanced_accuracy_score(yt, yp)
    t_macro_f1 = f1_score(yt, yp, average="macro", zero_division=0)
    t_weighted_f1 = f1_score(yt, yp, average="weighted", zero_division=0)
    return t_acc, t_bacc, t_macro_f1, t_weighted_f1, len(yt)


def run_fold(fold_name, stmae, X_pt, seq_idx_pt, packs, train_rows, test_rows):
    labels, s_sub, s_tri = packs[1:]
    groups = s_sub * 1000 + s_tri
    te_rows_pt = torch.tensor(test_rows, dtype=torch.long, device=DEVICE)

    # Strictly Single Seed (42)
    stmae_fold = copy.deepcopy(stmae)
    model, _ = fit(stmae_fold, X_pt, seq_idx_pt, labels, groups, train_rows, seed=RANDOM_SEED)
    if RECALIBRATE_BATCHNORM:
        model = adabn_recalibrate(model, X_pt, seq_idx_pt, te_rows_pt)
    probs = predict_probs(model, X_pt, seq_idx_pt, te_rows_pt)

    # Window-level Metrics
    yt = labels[test_rows]
    pred_win = probs.argmax(1)
    win_acc = accuracy_score(yt, pred_win)
    win_bacc = balanced_accuracy_score(yt, pred_win)
    win_macro_f1 = f1_score(yt, pred_win, average="macro", zero_division=0)
    win_weighted_f1 = f1_score(yt, pred_win, average="weighted", zero_division=0)

    # Trial-level Metrics
    t_acc, t_bacc, t_macro_f1, t_weighted_f1, num_trials = trial_level_scores(
        probs, yt, s_sub[test_rows], s_tri[test_rows]
    )

    print(
        f"  -> {fold_name:<11} | "
        f"WIN: acc={win_acc:.4f} bacc={win_bacc:.4f} macF1={win_macro_f1:.4f} wF1={win_weighted_f1:.4f} | "
        f"TRIAL: acc={t_acc:.4f} bacc={t_bacc:.4f} macF1={t_macro_f1:.4f} wF1={t_weighted_f1:.4f} "
        f"({len(test_rows)}w / {num_trials}t)"
    )

    REPORT.append(dict(
        fold=fold_name,
        win_acc=win_acc, win_bacc=win_bacc, win_macro_f1=win_macro_f1, win_weighted_f1=win_weighted_f1,
        trial_acc=t_acc, trial_bacc=t_bacc, trial_macro_f1=t_macro_f1, trial_weighted_f1=t_weighted_f1,
        num_windows=len(test_rows), num_trials=num_trials
    ))


def main():
    t0 = time.time()
    seed_everything(RANDOM_SEED)
    print("device:", DEVICE)
    print("Scalp regions:", {k: len(v) for k, v in REGIONS.items()})

    print("\n[1] loading DREAMER dataset (Target: VALENCE)")
    X_raw, Y_raw, subs, tri = load_or_extract_dreamer_valence()
    # Binary classification threshold on valence: > 3.0 = High Valence (1), <= 3.0 = Low Valence (0)
    y = (Y_raw > THRESHOLD).astype(np.int64)
    print(f"shape={X_raw.shape}  labels={np.bincount(y)} (0: Low Valence, 1: High Valence)")
    print("subjects:", sorted(np.unique(subs).tolist()))
    print("videos:", sorted(np.unique(tri).tolist()))

    print("\n[2] normalizing (dreamer notebook global Z-score)")
    X = normalize_dreamer_global(X_raw)

    print("\n[2.5] pushing dataset to VRAM for fast execution...")
    X_pt = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    print("\n[3] pretraining spatial masked autoencoder (STMAE) globally")
    stmae = pretrain_stmae(X_pt)
    for p in stmae.parameters():
        p.requires_grad = False

    print("\n[4] building sliding sequence windows")
    packs = build_sequence_index(y, subs, tri, WINDOW_LENGTH, stride=STRIDE)
    seq_idx = packs[0]
    seq_idx_pt = torch.tensor(seq_idx, dtype=torch.long, device=DEVICE)
    print(f"  {seq_idx.shape[0]} windows  labels={np.bincount(packs[1])}")

    # -----------------------------------------------------------------
    # LEAVE-ONE-SUBJECT-OUT (LOSO) EVALUATION
    # -----------------------------------------------------------------
    print("\n" + "=" * 84)
    print("EVALUATION: DREAMER VALENCE LOSO -- protocol=loso_subject (SEED = 42)")
    print("=" * 84)

    s_sub = packs[2]
    unique_subs = np.unique(s_sub)
    print(f"  loso_subject       {len(unique_subs)} folds x 1 seed = {len(unique_subs)} fits")

    for sb in unique_subs:
        te_rows = np.where(s_sub == sb)[0]
        tr_rows = np.where(s_sub != sb)[0]
        fold_name = f"subject_{sb + 1}"
        run_fold(fold_name, stmae, X_pt, seq_idx_pt, packs, tr_rows, te_rows)

    df = pd.DataFrame(REPORT)
    results_path = os.path.join(OUTPUT_DIR, "results_dreamer_valence_loso.csv")
    df.to_csv(results_path, index=False)
    print(f"\nsaved: {results_path}")

    print("\n" + "=" * 84)
    print("SUMMARY -- EVALUATION: DREAMER VALENCE LOSO (Single-Seed = 42)")
    print("=" * 84)
    if not df.empty:
        summary_dict = {
            "win_acc": [df["win_acc"].mean()],
            "win_bacc": [df["win_bacc"].mean()],
            "win_macro_f1": [df["win_macro_f1"].mean()],
            "win_weighted_f1": [df["win_weighted_f1"].mean()],
            "trial_acc": [df["trial_acc"].mean()],
            "trial_bacc": [df["trial_bacc"].mean()],
            "trial_macro_f1": [df["trial_macro_f1"].mean()],
            "trial_weighted_f1": [df["trial_weighted_f1"].mean()],
            "folds": [len(df)]
        }
        summ = pd.DataFrame(summary_dict)
        print(summ.to_string(index=False, float_format=lambda v: "%.4f" % v))

    print(f"\nWall time: {(time.time() - t0) / 60.0:.1f} min")


if __name__ == "__main__":
    main()

device: cuda
Scalp regions: {'frontal': 6, 'temporal_left': 2, 'parietal': 2, 'occipital': 2, 'temporal_right': 2}

[1] loading DREAMER dataset (Target: VALENCE)
  Parsing raw EEG from /kaggle/input/datasets/gautam2411/dreamer2/DREAMER.mat (Welch PSD + DE + Baseline Subtraction for Valence)...
shape=(171488, 14, 10)  labels=[101020  70468] (0: Low Valence, 1: High Valence)
subjects: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
videos: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

[2] normalizing (dreamer notebook global Z-score)

[2.5] pushing dataset to VRAM for fast execution...

[3] pretraining spatial masked autoencoder (STMAE) globally
  AE ep001 masked_recon_mse=0.50647
  AE ep005 masked_recon_mse=0.37930
  AE ep010 masked_recon_mse=0.36145
  AE ep015 masked_recon_mse=0.33701
  AE ep020 masked_recon_mse=0.32837
  AE ep025 masked_recon_mse=0.31917
  AE ep030 masked_recon_mse=0.30649

[4] building sliding sequence windows
  16